# Pre-Processing

### Install libraries & Load the dataset

In [10]:
import pandas as pd
import numpy as np
import os
import glob
from scipy import stats

## Dataset:
The depresjon dataset is a collection of motor activity data token from wrist actigraphy recorded from indivisuals, some are patients ( condition group ) diagnosed with bipolar, unipolar & depression, and some are healthy ( control group ).

The dataset contains 55 files:

23 Condition ( Patients )

32 Control ( Normal )

Scores file containing metadata & clinical / demographic data

In [3]:
SCORES = pd.read_csv("/content/drive/MyDrive/Main_Datasets/Depresjon/scores.csv")
CONDITION = glob.glob("/content/drive/MyDrive/Main_Datasets/Depresjon/condition/*.csv")
CONTROL = glob.glob("/content/drive/MyDrive/Main_Datasets/Depresjon/control/*.csv")

In [4]:
SCORES.head()

,number,days,gender,age,afftype,melanch,inpatient,edu,marriage,work,madrs1,madrs2
0,condition_1,11,2,35-39,2.0,2.0,2.0,6-10,1.0,2.0,19.0,19.0
1,condition_2,18,2,40-44,1.0,2.0,2.0,6-10,2.0,2.0,24.0,11.0
2,condition_3,13,1,45-49,2.0,2.0,2.0,6-10,2.0,2.0,24.0,25.0
3,condition_4,13,2,25-29,2.0,2.0,2.0,11-15,1.0,1.0,20.0,16.0
4,condition_5,13,2,50-54,2.0,2.0,2.0,11-15,2.0,2.0,26.0,26.0


In [5]:
def load_activity():
    frames = []
    for f in CONDITION:
        df = pd.read_csv(f)
        df['number'] = os.path.basename(f).replace(".csv","")
        df['group'] = "condition"
        frames.append(df)

    for f in CONTROL:
        df = pd.read_csv(f)
        df['number'] = os.path.basename(f).replace(".csv","")
        df['group'] = "control"
        frames.append(df)

    return pd.concat(frames, ignore_index=True)

activity = load_activity()
scores = SCORES

activity.head()

,timestamp,date,activity,number,group
0,2005-03-08 10:00:00,2005-03-08,0,condition_12,condition
1,2005-03-08 10:01:00,2005-03-08,0,condition_12,condition
2,2005-03-08 10:02:00,2005-03-08,0,condition_12,condition
3,2005-03-08 10:03:00,2005-03-08,3,condition_12,condition
4,2005-03-08 10:04:00,2005-03-08,0,condition_12,condition


## Date Formating

In [6]:
# Check out the formating of the dates: strict to the same date formats
activity['datetime'] = pd.to_datetime(
    activity['timestamp'],
    format="%Y-%m-%d %H:%M:%S",
    errors='coerce'
)
activity.head()

# Remove Timestamp & Date columns
activity = activity.drop(columns=['timestamp', 'date'])

## Resampling

In [7]:
# Determine Sampling Rate per Person
def get_sampling_rate(df):
    rates = {}
    for pid, g in df.groupby("number"):
        diff = g['datetime'].diff().dt.total_seconds().dropna()
        if len(diff) == 0:
            rates[pid] = None
        else:
            rates[pid] = diff.mode().iloc[0]
    return rates

sampling_rates = get_sampling_rate(activity)
list(sampling_rates.items())[:5]

[('condition_1', np.float64(60.0)),
 ('condition_10', np.float64(60.0)),
 ('condition_11', np.float64(60.0)),
 ('condition_12', np.float64(60.0)),
 ('condition_13', np.float64(60.0))]

In [8]:
# Choose a common sampling rate
common = np.array([30, 60, 300])
valid = [v for v in sampling_rates.values() if v is not None]

median_rate = np.median(valid)
TARGET_SEC = common[np.argmin(np.abs(common - median_rate))]

print("Chosen sampling rate:", TARGET_SEC)

Chosen sampling rate: 60


In [9]:
# Resample activity for each person
resampled_frames = []

for pid, g in activity.groupby("number"):
    g = g.set_index("datetime").sort_index()

    s = g['activity'].resample(f"{TARGET_SEC}S").mean()
    s = s.interpolate(limit=5)

    out = s.to_frame().reset_index()
    out['number'] = pid
    out['group'] = g['group'].iloc[0]
    resampled_frames.append(out)

resampled = pd.concat(resampled_frames, ignore_index=True)
resampled = resampled.dropna(subset=['activity'])

/tmp/ipython-input-2859399718.py:7: FutureWarning: 'S' is deprecated and will be removed in a future version, please use 's' instead.
  s = g['activity'].resample(f"{TARGET_SEC}S").mean()
/tmp/ipython-input-2859399718.py:7: FutureWarning: 'S' is deprecated and will be removed in a future version, please use 's' instead.
  s = g['activity'].resample(f"{TARGET_SEC}S").mean()
/tmp/ipython-input-2859399718.py:7: FutureWarning: 'S' is deprecated and will be removed in a future version, please use 's' instead.
  s = g['activity'].resample(f"{TARGET_SEC}S").mean()
/tmp/ipython-input-2859399718.py:7: FutureWarning: 'S' is deprecated and will be removed in a future version, please use 's' instead.
  s = g['activity'].resample(f"{TARGET_SEC}S").mean()
/tmp/ipython-input-2859399718.py:7: FutureWarning: 'S' is deprecated and will be removed in a future version, please use 's' instead.
  s = g['activity'].resample(f"{TARGET_SEC}S").mean()
/tmp/ipython-input-2859399718.py:7: FutureWarning: 'S' is de

## Removing Outliers
Extreme spikes in activity are removed using Z-score thresholding.

In [11]:
def remove_outliers(df, z_thresh=4):
    clean = []
    for pid, g in df.groupby('number'):
        z = np.abs(stats.zscore(g['activity'], nan_policy='omit'))
        clean.append(g[z < z_thresh])
    return pd.concat(clean, ignore_index=True)

cleaned = remove_outliers(resampled)

## Signal Smoothing
Rolling mean to reduce noise

In [12]:
cleaned['activity_smooth'] = cleaned.groupby('number')['activity'] \
    .transform(lambda x: x.rolling(3, min_periods=1).mean())

In [13]:
# Merge with scores
merged = cleaned.merge(scores, on="number", how="left")